# 10 · SC-FEGAN Fine-tuning (Phase 7-C1) ⭐

**목적**: SC-FEGAN G+D를 CelebAMask-HQ 2,000장으로 추가 학습 → 한국인 얼굴 도메인 적응.

**환경**: Colab T4 + **TF 2.x compat mode** (`TF_USE_LEGACY_KERAS=1`).

**산출물**: `/content/drive/MyDrive/cv-final-ckpts/scfegan_finetuned/`

**실패 시 폴백**: pretrained ckpt 그대로 사용 (학습 실패해도 발표 데모 가능).

## 1. 환경 셋업

In [ ]:
import os, sys
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/cv-final'
    if not os.path.exists(REPO):
        # private repo → Colab Secrets의 GH_TOKEN 사용
        from google.colab import userdata
        token = userdata.get('GH_TOKEN')
        !git clone https://{token}@github.com/keonU206/cv-final.git $REPO
    os.chdir(REPO)
else:
    REPO = os.path.abspath('..')
    os.chdir(REPO)

# Python import path에 cv-final 루트 추가
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# TF compat 환경변수는 import 전에 설정
os.environ['TF_USE_LEGACY_KERAS'] = '1'

if IS_COLAB:
    !pip install -q mediapipe==0.10.21 opencv-python pyyaml tf-keras
    !pip install -q 'tensorflow<2.16'  # legacy keras compat

print(f'CWD: {os.getcwd()}')

In [ ]:
# SC-FEGAN repo + pretrained ckpt 셋업
SCFEGAN_REPO = '/content/SC-FEGAN'
if IS_COLAB and not os.path.exists(SCFEGAN_REPO):
    !git clone https://github.com/run-youngjoo/SC-FEGAN.git $SCFEGAN_REPO

CKPT_PATH = '/content/drive/MyDrive/SC-FEGAN-ckpt/SC-FEGAN.ckpt'
print(f'pretrained ckpt exists: {os.path.exists(CKPT_PATH + ".index")}')

## 2. TF compat 동작 검증 (Smoke test)

In [ ]:
from project.scfegan_finetune.train_finetune import setup_tf_compat
ok = setup_tf_compat()
print(f'TF compat OK: {ok}')

## 3. 데이터 파이프라인 sanity check (1 sample)

In [ ]:
from project.scfegan_finetune.data_pipeline import (
    sample_celebamask_subset, build_finetune_sample,
)

paths = sample_celebamask_subset('/content/data/celebamask/images', n=10, seed=0)
print(f'sampled {len(paths)} paths')

pack = None
for p in paths:
    pack = build_finetune_sample(p, 'nose_tip', size=512, intensity=0.5)
    if pack is not None:
        break

if pack is None:
    print('⚠ 모든 이미지에서 랜드마크 추출 실패 — 데이터 확인 필요')
else:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(pack.image); axes[0].set_title('Source')
    axes[1].imshow(pack.composed['incomplete_image']); axes[1].set_title('Incomplete')
    axes[2].imshow(pack.composed['sketch'][..., 0], cmap='gray'); axes[2].set_title('Sketch')
    axes[3].imshow(pack.composed['color']); axes[3].set_title('Color')
    for ax in axes: ax.axis('off')
    plt.suptitle(f'Procedure: {pack.procedure_id}')
    plt.tight_layout(); plt.show()

## 4. Dry-run (1 step) → 환경 정상 확인

In [ ]:
from project.scfegan_finetune.train_finetune import train_finetune

dry_result = train_finetune(
    config_path='project/scfegan_finetune/config_finetune.yaml',
    dry_run=True,
)
print('Dry-run result:')
for k, v in dry_result.items():
    if k != 'history':
        print(f'  {k}: {v}')

## 5. Full 학습 (3 epochs)

⚠ 시간 약 4-6시간 예상 (Colab T4). 학습 도중 disconnect 대비:
- 체크포인트는 매 epoch마다 Drive에 저장됨
- 재시작 시 last epoch ckpt부터 이어 학습 가능

In [ ]:
if dry_result.get('used_fallback'):
    print(f"⚠ Dry-run에서 fallback 발생: {dry_result.get('fallback_reason')}")
    print('  → Full 학습 skip. pretrained ckpt 그대로 사용.')
else:
    full_result = train_finetune(
        config_path='project/scfegan_finetune/config_finetune.yaml',
        dry_run=False,
    )
    print('Full 학습 완료:')
    for k, v in full_result.items():
        if k != 'history':
            print(f'  {k}: {v}')

## 6. 결과 검증 — Pretrained vs Fine-tuned 비교

동일 입력에 대해 두 ckpt로 inference 후 시각화. 별도 notebook(`11_pretrained_vs_finetuned.ipynb`)에서 진행하거나, 본 노트북 마지막 셀에서 간단 비교.

In [ ]:
print('학습 완료. Phase 7-C1 다음 단계:')
print('  1. notebooks/08_scfegan_integration.ipynb 재실행 — fine-tuned ckpt로 inference')
print('  2. notebooks/11_pretrained_vs_finetuned.ipynb 작성 — Before/After 비교')
print('  3. 발표 슬라이드에 결과 추가')